# Knowledge Distillation Notebook

## Setup

In [1]:
%%capture
!pip install unsloth
!pip install --upgrade trl datasets

from unsloth import FastLanguageModel
import torch
import json

## Bronze Level: Teacher generiert Trainingsdaten

## Step 1A: Load Base Model 
- Load base model from Huggingface

In [2]:
from unsloth import FastLanguageModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## Step 1B: Apply Adapter to Base Model

Either use default QLora or Fine-Tuned Adapter from Huggingface

In [3]:
from peft import PeftModel

# Loads Qlora and applies to model
if False:
    if 'model' in dir() and isinstance(model, PeftModel):
        print("Unloading existing Adapter...")
        model = model.unload()
    else:
        pass
        
    model = FastLanguageModel.get_peft_model(
        base_model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
    # Check if Adapter was applied successfully
    if isinstance(model, PeftModel):
        print("Adapter applied successfully!")
    else:
        print("Adapter was not found")


# Load fine tuned adapter from Huggingface and applies to model weights
if True:
    if 'model' in dir() and isinstance(model, PeftModel):
        print("Unloading existing Adapter...")
        model = model.unload()
    else:
        pass
    
    model = PeftModel.from_pretrained(
        base_model,
        "Feyerade/german-support-qwen-lora-adapter",
    )
    # Check if Adapter was applied successfully
    if isinstance(model, PeftModel):
        print("Adapter applied successfully!")
    else:
        print("Adapter was not found")

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/120M [00:00<?, ?B/s]

Adapter applied successfully!


## Set 2: Set Teacher to inference mode

In [4]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

### Step 3: Define prompt

In [5]:
prompts = [
    "Meine Lieferung sollte gestern ankommen, aber im Tracking steht immer noch 'in Bearbeitung'.",
    "Ich kann mich nicht mehr in mein Konto einloggen. Der Code per Email kommt nicht an.",
    "Warum wurde meine Bestellung ohne Grund storniert?",
    "Ich habe eine Versandbestaetigung bekommen, aber keine Sendungsnummer.",
    "Kann ich meine Bestellung noch an eine andere Adresse schicken lassen?",
    "Ich habe aus Versehen zwei Mal bestellt. Koennen Sie eine Bestellung stornieren?",
    "Der Preis im Warenkorb ist hoeher als auf der Produktseite.",
    "Warum wurde meine Zahlung abgelehnt obwohl genug Geld auf der Karte ist?",
    "Ich finde meine alte Bestellung in meinem Konto nicht mehr.",
    "Wie lange dauert der Versand normalerweise?",
    "Ich habe keine Bestaetigungsmail fuer meine Bestellung erhalten.",
    "Der Rabattcode aus dem Newsletter funktioniert nicht.",
    "Kann ich meine Bestellung auch an eine Packstation liefern lassen?",
    "Meine Bestellung ist laut Tracking zugestellt, aber ich habe nichts erhalten.",
    "Wie kann ich meine Zahlungsmethode aendern?",
    "Ich moechte eine Kopie meiner letzten Rechnung herunterladen.",
    "Warum wurde meine Bestellung in zwei Paketen verschickt?",
    "Das Produkt sieht anders aus als auf der Website.",
    "Ich kann meine Bestellung nicht abschliessen. Der Bezahlbutton reagiert nicht.",
    "Warum sind die Versandkosten ploetzlich hoeher als vorher?",
    "Ich habe eine Mahnung bekommen, obwohl ich schon bezahlt habe.",
    "Wie kann ich mein Passwort wiederherstellen wenn ich keinen Zugriff auf meine Email habe?",
    "Mein Warenkorb ist nach dem Einloggen leer.",
    "Kann ich mehrere Gutscheine gleichzeitig einloesen?",
    "Ich habe versehentlich die falsche Farbe bestellt. Kann ich das noch aendern?",
    "Der Kundenservice antwortet seit Tagen nicht auf meine Email.",
    "Ich moechte wissen wann meine Rueckerstattung bearbeitet wird.",
    "Meine Bestellung wurde als geliefert markiert, aber ich habe kein Paket bekommen.",
    "Warum funktioniert die Zahlung mit PayPal nicht?",
    "Ich moechte mein Lieferdatum verschieben.",
    "Kann ich meine Bestellung auch selbst im Laden abholen?",
    "Das Paket ist beschaedigt angekommen und der Inhalt fehlt teilweise.",
    "Ich habe ein Abo abgeschlossen, aber finde die Einstellungen nicht.",
    "Warum wird meine Kreditkarte nicht akzeptiert?",
    "Ich habe eine falsche Rechnung bekommen.",
    "Der Artikel im Paket entspricht nicht meiner Bestellung.",
    "Warum kann ich mein Konto nicht verifizieren?",
    "Ich moechte mein Passwort aendern, finde aber die Option nicht.",
    "Meine Bestellung ist seit Tagen im Status 'wird vorbereitet'.",
    "Ich habe einen Artikel zurueckgeschickt, aber noch keine Rueckerstattung erhalten.",
    "Warum kann ich keine Bewertung fuer ein Produkt abgeben?",
    "Ich habe eine Email ueber eine Bestellung erhalten die ich nicht gemacht habe.",
    "Der Support Chat laedt nicht in meinem Browser.",
    "Ich kann mein Profilbild nicht hochladen.",
    "Warum wurde mein Konto ohne Vorwarnung gesperrt?",
    "Ich finde die Option zum Abbestellen des Newsletters nicht.",
    "Die Website zeigt eine Fehlermeldung beim Checkout.",
    "Mein Gutschein wurde als benutzt markiert obwohl ich ihn nicht verwendet habe.",
    "Ich moechte meine Lieferadresse dauerhaft aendern.",
    "Warum sind manche Produkte in meinem Land nicht lieferbar?",
    "Ich habe den falschen Artikel erhalten.",
    "Meine Bestellung wurde teilweise geliefert. Wann kommt der Rest?",
    "Warum wurde meine Ruecksendung abgelehnt?",
    "Ich kann die App nicht auf meinem Handy installieren.",
    "Die App zeigt nur einen weissen Bildschirm beim Start.",
    "Ich bekomme staendig Fehlermeldungen beim Einloggen.",
    "Wie kann ich mein Kundenkonto komplett loeschen?",
    "Ich moechte meine Emailadresse im Konto aendern.",
    "Warum funktioniert die Zwei Faktor Anmeldung nicht?",
    "Ich habe meine Bestellung an die falsche Adresse geschickt.",
    "Wie kann ich einen Artikel umtauschen?",
    "Mein Paket ist im Versand verloren gegangen.",
    "Ich sehe eine unbekannte Belastung auf meiner Rechnung.",
    "Kann ich eine Bestellung pausieren bevor sie verschickt wird?",
    "Warum ist mein Konto ploetzlich deaktiviert?",
    "Ich kann keine neuen Produkte in den Warenkorb legen.",
    "Der Warenkorb aktualisiert die Menge nicht richtig.",
    "Ich habe einen Gutschein geschenkt bekommen, aber er wird nicht akzeptiert.",
    "Kann ich eine Ruecksendung ohne Originalverpackung machen?",
    "Mein Paket wurde an den Absender zurueckgeschickt.",
    "Ich finde die Rechnung fuer meine letzte Bestellung nicht.",
    "Warum ist meine Bestellung teurer als erwartet?",
    "Wie lange dauert eine Rueckerstattung normalerweise?",
    "Ich moechte wissen ob ein Produkt wieder auf Lager kommt.",
    "Warum funktioniert der Login ueber Google nicht?",
    "Ich bekomme keine SMS fuer die Anmeldung.",
    "Mein Konto zeigt eine falsche Bestellhistorie.",
    "Ich kann meine Bestellung nicht verfolgen.",
    "Warum wird mein Gutschein nicht auf reduzierte Artikel angewendet?",
    "Ich habe eine falsche Emailadresse im Konto gespeichert.",
    "Wie kann ich meine Bestellung komplett stornieren?",
    "Warum funktioniert der Warenkorb auf dem Handy nicht?",
    "Ich sehe doppelte Bestellungen in meinem Konto.",
    "Der Support Bot versteht meine Anfrage nicht.",
    "Meine Bestellung wurde zweimal berechnet.",
    "Warum kann ich kein neues Passwort setzen?",
    "Ich habe eine Lieferung an eine alte Adresse bekommen.",
    "Ich moechte mein Abo vorzeitig beenden.",
    "Warum wurde mein Gutschein deaktiviert?",
    "Ich kann meine Ruecksendung im Portal nicht anmelden.",
    "Der Artikel fehlt in meiner Lieferung.",
    "Meine Bestellung wurde automatisch storniert.",
    "Ich moechte eine Rechnung mit ausgewiesener Mehrwertsteuer.",
    "Warum funktioniert die Zahlungsseite nicht?",
    "Ich habe eine falsche Telefonnummer im Konto hinterlegt.",
    "Kann ich eine Rueckerstattung auf eine andere Zahlungsmethode bekommen?",
    "Mein Paket wurde beim Nachbarn abgegeben ohne Info.",
    "Warum bekomme ich keine Versandbestaetigung?",
    "Ich habe einen falschen Namen auf der Rechnung.",
    "Kann ich meine Bestellung beschleunigen?",
    "Der Artikel ist defekt angekommen.",
    "Warum ist mein Konto ploetzlich leer?",
    "Ich kann meine Lieferadresse nicht speichern.",
    "Die Website ist sehr langsam beim Bestellen.",
    "Mein Rabattcode ist angeblich abgelaufen.",
    "Ich moechte mein Abo auf monatliche Zahlung umstellen.",
    "Warum wird meine Bestellung immer wieder abgelehnt?",
    "Ich kann meine Zahlungsart nicht entfernen.",
    "Mein Paket steckt seit Tagen im Versandzentrum fest.",
    "Ich habe ein Produkt bestellt das jetzt ausverkauft ist.",
    "Warum bekomme ich staendig Fehlermeldungen im Konto?",
    "Ich kann meine Daten im Profil nicht bearbeiten.",
    "Mein Warenkorb verschwindet nach dem Aktualisieren.",
    "Ich moechte eine Sendung umleiten.",
    "Warum sehe ich unterschiedliche Preise fuer das gleiche Produkt?",
    "Ich habe meine Bestellung aus Versehen bestaetigt.",
    "Kann ich meine Bestellung nach dem Versand noch stornieren?",
    "Mein Gutschein funktioniert nur teilweise.",
    "Ich bekomme staendig Werbung obwohl ich mich abgemeldet habe.",
    "Wie kann ich mein Konto voruebergehend deaktivieren?",
    "Meine Rueckerstattung ist noch nicht auf dem Konto.",
    "Ich kann meine Bestellung nicht herunterladen.",
    "Warum funktioniert die Filterfunktion im Shop nicht?",
    "Ich habe eine falsche Groesse geliefert bekommen.",
    "Mein Paket wurde geoeffnet geliefert.",
    "Ich moechte meine Zahlungsart auf Rechnung aendern.",
    "Warum kann ich keine Artikel bewerten?",
    "Ich sehe eine unbekannte Bestellung in meinem Konto.",
    "Meine Bestellung ist verschwunden.",
    "Ich kann meine Telefonnummer nicht bestaetigen.",
    "Der Preis im Checkout ist falsch berechnet.",
    "Ich moechte eine Bestellung zusammenlegen.",
    "Warum funktioniert der Warenkorb nicht im Browser?",
    "Ich kann meine Ruecksendung nicht ausdrucken.",
    "Meine Lieferung kam viel spaeter als angekuendigt.",
    "Warum kann ich keinen neuen Gutschein einloesen?",
    "Ich habe keine Rechnung per Email erhalten.",
    "Der Trackinglink funktioniert nicht.",
    "Ich kann meine Bestellung nicht bearbeiten.",
    "Warum wurde mein Konto aus Sicherheitsgruenden gesperrt?",
    "Ich moechte meine Daten exportieren.",
    "Der Artikel kam ohne Zubehoer.",
    "Warum ist mein Konto ploetzlich ausgeloggt?",
    "Ich kann die App nicht aktualisieren.",
    "Meine Bestellung wurde aufgeteilt ohne Info.",
    "Ich moechte eine Teilerstattung fuer einen Artikel.",
    "Warum wurde meine Ruecksendung noch nicht bestaetigt?",
    "Ich sehe eine falsche Adresse im Konto.",
    "Mein Paket wurde beim Transport beschaedigt.",
    "Ich moechte den Versand auf Express aendern.",
    "Warum kann ich kein neues Konto erstellen?",
    "Ich bekomme keine Aktivierungsmail.",
    "Der Login Button funktioniert nicht.",
    "Ich habe eine Lieferung fuer jemand anderen erhalten.",
    "Warum wird mein Gutschein nur teilweise angewendet?",
    "Ich moechte eine Bestellung fuer meine Firma machen.",
    "Meine Bestellung wurde ohne Grund zurueckgesendet.",
    "Ich kann mein Passwort nicht zuruecksetzen.",
    "Warum funktioniert die Suche im Shop nicht?",
    "Ich sehe einen anderen Preis in der App.",
    "Meine Bestellung wurde doppelt verschickt.",
    "Ich moechte meine Bestellung splitten.",
    "Warum ist mein Konto eingeschraenkt?",
    "Ich bekomme keine Updates zum Versand.",
    "Meine Ruecksendung wurde noch nicht bearbeitet.",
    "Ich kann mein Profil nicht speichern.",
    "Warum funktioniert die Zahlungsbestaetigung nicht?",
    "Mein Paket ist leer angekommen.",
    "Ich moechte meine Bestellung an einen anderen Empfaenger schicken.",
    "Warum funktioniert der Gutscheincode nicht im Warenkorb?",
    "Ich kann meine Adresse nicht bestaetigen.",
    "Meine Bestellung wurde ohne Info geaendert.",
    "Warum sehe ich falsche Preise im Warenkorb?",
    "Ich moechte eine Bestellung verschenken.",
    "Meine Rechnung zeigt falsche Positionen.",
    "Warum kann ich mein Konto nicht loeschen?",
    "Ich habe eine Bestellung ohne Konto gemacht und finde sie nicht.",
    "Der Artikel ist ploetzlich nicht mehr verfuegbar.",
    "Ich kann meine Ruecksendung nicht verfolgen.",
    "Warum funktioniert der Login in der App nicht?",
    "Ich moechte eine Bestellung fuer spaeter planen.",
    "Mein Paket wurde im Regen abgestellt.",
    "Warum funktioniert mein Gutscheincode nur einmal?",
    "Ich sehe eine fremde Adresse im Konto.",
    "Meine Bestellung wurde ohne Zustimmung storniert.",
    "Ich kann mein Konto nicht wieder aktivieren.",
    "Warum funktioniert der Support Chat nicht?",
    "Meine Lieferung wurde an die falsche Adresse geschickt.",
    "Ich moechte meine Bestellung neu berechnen lassen.",
    "Warum kann ich keinen Rabatt anwenden?",
    "Ich sehe keine Trackinginformationen.",
    "Meine Ruecksendung wurde verloren.",
    "Ich kann meine Bestellung nicht herunterladen.",
    "Warum wird mein Warenkorb nicht gespeichert?",
    "Ich habe eine falsche Bestellbestaetigung erhalten.",
    "Mein Paket wurde im Treppenhaus abgestellt.",
    "Warum funktioniert mein Login Code nicht?",
    "Ich moechte eine Bestellung fuer jemand anderen bezahlen.",
    "Meine Rechnung zeigt eine doppelte Belastung.",
    "Warum kann ich kein neues Passwort erstellen?",
    "Ich habe eine Bestellung bekommen die ich nicht bestellt habe.",
    "Mein Gutschein wurde abgelehnt.",
    "Warum funktioniert der Checkout nicht?",
    "Ich kann meine Bestellung nicht sehen.",
    "Meine Ruecksendung wurde nicht akzeptiert.",
    "Warum kann ich meine Adresse nicht aktualisieren?",
    "Ich habe einen Artikel doppelt bekommen.",
    "Meine Bestellung wurde ohne Grund verzzoegert.",
    "Warum bekomme ich keine Email bestaetigung?",
    "Ich kann mein Konto nicht verifizieren.",
    "Meine Bestellung wurde falsch gepackt.",
    "Warum funktioniert mein Rabatt nicht?",
    "Ich sehe eine unbekannte Zahlung.",
    "Meine Lieferung fehlt komplett.",
    "Warum wurde mein Konto deaktiviert?",
    "Ich kann meine Bestellung nicht abschliessen.",
    "Meine Rueckerstattung ist zu niedrig.",
    "Warum funktioniert mein Gutschein nicht mehr?",
    "Ich sehe eine falsche Lieferadresse.",
    "Meine Bestellung ist verschwunden.",
    "Warum kann ich keine Zahlung abschliessen?",
    "Ich habe eine falsche Rechnung bekommen.",
    "Meine Lieferung ist unvollstaendig.",
    "Warum funktioniert der Login nicht mehr?",
    "Ich kann mein Passwort nicht aendern.",
    "Meine Bestellung wurde storniert ohne Info.",
    "Warum sehe ich falsche Preise?",
    "Ich kann meine Bestellung nicht verfolgen.",
    "Meine Ruecksendung wurde nicht bestaetigt.",
    "Warum funktioniert der Gutschein nicht?",
    "Ich kann mein Konto nicht finden.",
    "Meine Bestellung wurde doppelt berechnet."
]

## Step 4: Generate Answers with Teacher Model

In [6]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning) # Disable future warnings

import transformers
transformers.logging.set_verbosity_error() # Sets transformers to only display errors (not warnings)

generated_data = []

system_prompt = """Du bist ein professioneller Kundenservice-Mitarbeiter.
Antworte freundlich, loesungsorientiert und auf Deutsch.
Halte deine Antworten unter 150 Woertern."""

for i, prompt in enumerate(prompts):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    generated_data.append({
        "instruction": prompt,
        "output": response,
    })

    if (i + 1) % 25 == 0:
        print(f"Generiert: {i + 1}/{len(prompts)}")

print(f"\nFertig. {len(generated_data)} Antworten generiert.")

Generiert: 25/232
Generiert: 50/232
Generiert: 75/232
Generiert: 100/232
Generiert: 125/232
Generiert: 150/232
Generiert: 175/232
Generiert: 200/232
Generiert: 225/232

Fertig. 232 Antworten generiert.


### Step 5: Save Dataset

In [9]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

with open("teacher_generated_data.json", "w", encoding="utf-8") as f:
    json.dump(generated_data, f, ensure_ascii=False, indent=2)

print(f"Written: teacher_generated_data.json ({len(generated_data)} entries)")

from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="teacher_generated_data.json",
    path_in_repo="teacher_generated_data.json",
    repo_id="Feyerade/german-support-qwen-lora-adapter",  # dein existierendes Repo
    token=HF_TOKEN,
)
print("Uploaded to HuggingFace.")

Written: teacher_generated_data.json (232 entries)
Uploaded to HuggingFace.


## Print generated answers to check quality

In [10]:
# Stichprobe anschauen
for item in generated_data[:15]:
    print(f"\n{'='*60}")
    print(f"FRAGE: {item['instruction']}")
    print(f"ANTWORT: {item['output']}")


FRAGE: Meine Lieferung sollte gestern ankommen, aber im Tracking steht immer noch 'in Bearbeitung'.
ANTWORT: Das ist aergerlich! Eine verlorene Bestellung ist natuerlich nicht akzeptabel.

Bitte teilen Sie mir Ihre Bestellnummer mit. Wir klaeren das sofort:
1. Kontaktieren Sie uns mit 'verlorene Bestellung'
2. Senden wir eine E-Mail mit einer kostenlosen Ruecksendung oder Ersatz
3. Sollte die Ware verloren gegangen sein, senden wir Ihnen Geldueberhang


FRAGE: Ich kann mich nicht mehr in mein Konto einloggen. Der Code per Email kommt nicht an.
ANTWORT: Das ist frustrierend — lassen Sie uns das gemeinsam loesen!

Bitte versuchen Sie nochmals mit Ihrem Zwillingskonto — das sind die letzten beiden Ziffern Ihres Telefons:
123 oder 321

Sollte das auch fehlschlagen, sperren wir Ihr Konto voruebergehend und entsperren es sicher fuer Sie.

FRAGE: Warum wurde meine Bestellung ohne Grund storniert?
ANTWORT: Das kann verschiedene Gründen haben:

1. Falsches Sendungsverfolgungsnummer — Wir pruef

## Silver Level: Student fine-tunen

## Step 1: Remove Teacher model

In [11]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

print("Teacher entladen. GPU-Speicher freigegeben.")

Teacher entladen. GPU-Speicher freigegeben.


## Step 2: Load Student model (1.5B)

In [12]:
student_model, student_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

student_model = FastLanguageModel.get_peft_model(
    student_model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("Student geladen: Qwen2.5-1.5B-Instruct")

==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Student geladen: Qwen2.5-1.5B-Instruct


## Step 3: Load teacher-generated dataset

In [15]:
from datasets import Dataset
from huggingface_hub import hf_hub_download

# load JSON from HuggingFace
hf_hub_download(
    repo_id="Feyerade/german-support-qwen-lora-adapter",
    filename="teacher_generated_data.json",
    local_dir=".",
    token=HF_TOKEN,
)


with open("teacher_generated_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# In das Chat-Format bringen das Unsloth erwartet
def format_for_training(item):
    messages = [
        {"role": "system", "content": "Du bist ein professioneller Kundenservice-Mitarbeiter. Antworte freundlich, loesungsorientiert und auf Deutsch."},
        {"role": "user", "content": item["instruction"]},
        {"role": "assistant", "content": item["output"]},
    ]
    text = student_tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_for_training)

print(f"Dataset: {len(dataset)} Eintraege")
print(f"Beispiel:\n{dataset[0]['text'][:300]}...")

Map:   0%|          | 0/232 [00:00<?, ? examples/s]

Dataset: 232 Eintraege
Beispiel:
<|im_start|>system
Du bist ein professioneller Kundenservice-Mitarbeiter. Antworte freundlich, loesungsorientiert und auf Deutsch.<|im_end|>
<|im_start|>user
Meine Lieferung sollte gestern ankommen, aber im Tracking steht immer noch 'in Bearbeitung'.<|im_end|>
<|im_start|>assistant
Das ist aergerlic...


## Step 4: Train Student model

In [16]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=student_model,
    tokenizer=student_tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="student_outputs",
    ),
)

print("Training startet...")
trainer_stats = trainer.train()
print(f"Training fertig. Final Loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/232 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Training startet...
{'loss': '2.984', 'grad_norm': '1.087', 'learning_rate': '0.00016', 'epoch': '0.1724'}
{'loss': '2.115', 'grad_norm': '1.415', 'learning_rate': '0.0001855', 'epoch': '0.3448'}
{'loss': '1.643', 'grad_norm': '0.8073', 'learning_rate': '0.0001673', 'epoch': '0.5172'}
{'loss': '1.486', 'grad_norm': '0.6627', 'learning_rate': '0.0001491', 'epoch': '0.6897'}
{'loss': '1.38', 'grad_norm': '0.7277', 'learning_rate': '0.0001309', 'epoch': '0.8621'}
{'loss': '1.296', 'grad_norm': '0.6666', 'learning_rate': '0.0001127', 'epoch': '1.034'}
{'loss': '1.203', 'grad_norm': '0.7481', 'learning_rate': '9.455e-05', 'epoch': '1.207'}
{'loss': '1.151', 'grad_norm': '0.7032', 'learning_rate': '7.636e-05', 'epoch': '1.379'}
{'loss': '1.07', 'grad_norm': '0.7557', 'learning_rate': '5.818e-05', 'epoch': '1.552'}
{'loss': '1.101', 'grad_norm': '0.781', 'learning_rate': '4e-05', 'epoch': '1.724'}
{'loss': '1.126', 'grad_norm': '

### Save Destilled Student model to Huggingface

In [18]:
student_model.push_to_hub(
    "Feyerade/german-support-student-1.5b-distilled",
    token=HF_TOKEN,
)
student_tokenizer.push_to_hub(
    "Feyerade/german-support-student-1.5b-distilled",
    token=HF_TOKEN,
)
print("Student model and Tokenizer pushed to HuggingFace.")

README.md:   0%|          | 0.00/566 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Feyerade/german-support-student-1.5b-distilled


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Student model pushed to HuggingFace.


## Silver challenge:

- Finaler loss: 1.054099


## Gold Challenge: